In [0]:
import math

CATALOG   = "e-commerce"
SILVER_DB = "silver"
GOLD_DB   = "gold"

In [0]:
spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")
print("Gold database ready")

In [0]:
spark.table(f"{SILVER_DB}.fact_sales").createOrReplaceTempView("fact_sales")
spark.table(f"{SILVER_DB}.dim_calendar").createOrReplaceTempView("dim_calendar")
spark.table(f"{SILVER_DB}.dim_customers").createOrReplaceTempView("dim_customers")
spark.table(f"{SILVER_DB}.dim_products").createOrReplaceTempView("dim_products")
spark.table(f"{SILVER_DB}.dim_stores").createOrReplaceTempView("dim_stores")

print("Silver views registered")

In [0]:
df_revenue_by_month = spark.sql("""
    SELECT
        c.year,
        c.quarter,
        c.quarter_name,
        c.year_quarter,
        c.month,
        c.month_name,
        c.month_short,
        c.year_month,

        COUNT(f.order_id)                               AS total_orders,
        SUM(f.quantity)                                 AS total_units_sold,

        ROUND(SUM(f.gross_sales),  2)                   AS gross_sales,
        ROUND(SUM(f.discount_amount), 2)                AS total_discount,
        ROUND(SUM(f.revenue),      2)                   AS total_revenue,
        ROUND(AVG(f.revenue),      2)                   AS avg_order_revenue,

        ROUND(SUM(f.cost),         2)                   AS total_cost,
        ROUND(SUM(f.profit),       2)                   AS total_profit,
        ROUND(AVG(f.profit_margin) * 100, 2)            AS avg_profit_margin_pct,

        ROUND(SUM(f.revenue) / COUNT(f.order_id), 2)   AS aov

    FROM fact_sales   f
    JOIN dim_calendar c ON f.date_sk = c.date_sk
    WHERE f.is_valid = 1
    GROUP BY
        c.year, c.quarter, c.quarter_name, c.year_quarter,
        c.month, c.month_name, c.month_short, c.year_month
    ORDER BY c.year, c.month
""")

df_revenue_by_month.show(24, truncate=False)

row_count = df_revenue_by_month.count()
num_partitions = max(2, math.ceil(row_count / 10000))

df_revenue_by_month.repartition(num_partitions).write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.revenue_by_month")

print(f"gold.revenue_by_month written - {row_count} rows ({num_partitions} partitions)")

In [0]:
df_daily_sales = spark.sql("""
    SELECT
        c.date,
        c.year,
        c.month,
        c.month_name,
        c.week,
        c.day_name,
        c.day_type,
        c.is_weekend,

        COUNT(f.order_id)                AS daily_orders,
        SUM(f.quantity)                  AS daily_units,
        ROUND(SUM(f.revenue),   2)       AS daily_revenue,
        ROUND(SUM(f.profit),    2)       AS daily_profit,
        ROUND(AVG(f.profit_margin)*100,2) AS avg_margin_pct,
        ROUND(SUM(f.revenue) / COUNT(f.order_id), 2) AS daily_aov

    FROM fact_sales   f
    JOIN dim_calendar c ON f.date_sk = c.date_sk
    WHERE f.is_valid = 1
    GROUP BY
        c.date, c.year, c.month, c.month_name,
        c.week, c.day_name, c.day_type, c.is_weekend
    ORDER BY c.date
""")

df_daily_sales.show(10, truncate=False)

row_count = df_daily_sales.count()
num_partitions = max(2, math.ceil(row_count / 10000))

df_daily_sales.repartition(num_partitions).write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.daily_sales_trend")

print(f"gold.daily_sales_trend written - {row_count} rows ({num_partitions} partitions)")

In [0]:
df_yoy = spark.sql("""
    WITH yearly AS (
        SELECT
            c.year,
            COUNT(f.order_id)          AS total_orders,
            SUM(f.quantity)            AS total_units,
            ROUND(SUM(f.revenue),  2)  AS total_revenue,
            ROUND(SUM(f.profit),   2)  AS total_profit,
            ROUND(AVG(f.profit_margin)*100, 2) AS avg_margin_pct
        FROM fact_sales   f
        JOIN dim_calendar c ON f.date_sk = c.date_sk
        WHERE f.is_valid = 1
        GROUP BY c.year
    ),
    yoy AS (
        SELECT
            curr.year,
            curr.total_orders,
            curr.total_units,
            curr.total_revenue,
            curr.total_profit,
            curr.avg_margin_pct,
            prev.total_revenue                              AS prev_year_revenue,
            prev.total_profit                               AS prev_year_profit,
            prev.total_orders                               AS prev_year_orders,

            ROUND(
                (curr.total_revenue - prev.total_revenue)
                / prev.total_revenue * 100, 2
            )                                               AS revenue_growth_pct,

            ROUND(
                (curr.total_profit - prev.total_profit)
                / prev.total_profit * 100, 2
            )                                               AS profit_growth_pct,

            ROUND(
                (curr.total_orders - prev.total_orders)
                / prev.total_orders * 100, 2
            )                                               AS order_growth_pct

        FROM yearly curr
        LEFT JOIN yearly prev ON curr.year = prev.year + 1
    )
    SELECT * FROM yoy ORDER BY year
""")

df_yoy.show(truncate=False)

row_count = df_yoy.count()
num_partitions = max(2, math.ceil(row_count / 10000))

df_yoy.repartition(num_partitions).write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.yoy_growth")

print(f"gold.yoy_growth written - {row_count} rows")

In [0]:
df_aov = spark.sql("""
    SELECT
        c.year,
        c.year_quarter,
        c.year_month,
        c.month_name,

        COUNT(f.order_id)                                AS total_orders,
        ROUND(SUM(f.revenue), 2)                         AS total_revenue,
        ROUND(SUM(f.revenue) / COUNT(f.order_id), 2)    AS aov,
        ROUND(AVG(f.quantity), 2)                        AS avg_units_per_order,
        ROUND(AVG(f.unit_price), 2)                      AS avg_unit_price,
        ROUND(AVG(f.discount)*100, 2)                    AS avg_discount_pct

    FROM fact_sales   f
    JOIN dim_calendar c ON f.date_sk = c.date_sk
    WHERE f.is_valid = 1
    GROUP BY c.year, c.year_quarter, c.year_month, c.month_name
    ORDER BY c.year, c.year_month
""")

df_aov.show(24, truncate=False)

row_count = df_aov.count()
num_partitions = max(2, math.ceil(row_count / 10000))

df_aov.repartition(num_partitions).write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.aov_summary")

print(f"gold.aov_summary written - {row_count} rows")

In [0]:
df_day_type = spark.sql("""
    SELECT
        c.day_name,
        c.day_of_week,
        c.day_type,
        c.is_weekend,

        COUNT(f.order_id)                               AS total_orders,
        SUM(f.quantity)                                 AS total_units,
        ROUND(SUM(f.revenue),  2)                       AS total_revenue,
        ROUND(SUM(f.profit),   2)                       AS total_profit,
        ROUND(AVG(f.revenue),  2)                       AS avg_daily_revenue,
        ROUND(SUM(f.revenue) / COUNT(f.order_id), 2)   AS aov,
        ROUND(AVG(f.profit_margin)*100, 2)              AS avg_margin_pct

    FROM fact_sales   f
    JOIN dim_calendar c ON f.date_sk = c.date_sk
    WHERE f.is_valid = 1
    GROUP BY c.day_name, c.day_of_week, c.day_type, c.is_weekend
    ORDER BY c.day_of_week
""")

df_day_type.show(truncate=False)

print("Weekday vs Weekend summary")
spark.sql("""
    SELECT
        c.day_type,
        COUNT(f.order_id)                               AS total_orders,
        ROUND(SUM(f.revenue), 2)                        AS total_revenue,
        ROUND(AVG(f.revenue), 2)                        AS avg_order_revenue,
        ROUND(SUM(f.profit),  2)                        AS total_profit,
        ROUND(AVG(f.profit_margin)*100, 2)              AS avg_margin_pct
    FROM fact_sales   f
    JOIN dim_calendar c ON f.date_sk = c.date_sk
    WHERE f.is_valid = 1
    GROUP BY c.day_type
    ORDER BY c.day_type
""").show(truncate=False)

row_count = df_day_type.count()
num_partitions = max(2, math.ceil(row_count / 10000))

df_day_type.repartition(num_partitions).write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{GOLD_DB}.sales_by_day_type")

print(f"gold.sales_by_day_type written - {row_count} rows")

In [0]:
print("GOLD - Revenue & Sales COMPLETE")
print("  gold.revenue_by_month    - Revenue by month/yr")
print("  gold.daily_sales_trend   - Daily trend")
print("  gold.yoy_growth          - YoY growth")
print("  gold.aov_summary         - AOV by month")
print("  gold.sales_by_day_type   - Weekday vs weekend")
print()

for tbl in ["revenue_by_month", "daily_sales_trend", "yoy_growth", "aov_summary", "sales_by_day_type"]:
    cnt = spark.table(f"{GOLD_DB}.{tbl}").count()
    print(f"   gold.{tbl:<25} - {cnt:,} rows")